## Homework 6 - Supervised finetuning for chat and DPO

In this homework, you will finetune a pretrained model (of the _exact_ same model cdoe you developed in the previous assignment, just trained for longer) for chat training, using both supervised finetuning (SFT) and direct preference optimization (DPO).

In [1]:
### Run this cell to download and install the necessary modules for the homework
!pip install --upgrade --no-deps git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/modernaicourse/hw6/refs/heads/main/hw6_tests.py

import os
import math
import mugrade
import torch
import tiktoken
from torch.nn import Module, ModuleList, Parameter, Buffer
import json
import copy


from hw6_tests import (test_Linear,
                       test_Embedding,
                       test_silu,
                       test_rms_norm,
                       test_self_attention,
                       test_MultiHeadAttentionKVCache,
                       test_MLP,
                       test_TransformerBlock,
                       test_cross_entropy_loss,
                       test_Adam,
                       test_LLM,
                       test_generate,
                       test_convert_to_chat_format, submit_convert_to_chat_format,
                       test_pretokenize_chat, submit_pretokenize_chat,
                       test_get_loss_mask, submit_get_loss_mask,
                       test_DataLoaderChat, submit_DataLoaderChat,
                       test_train_llm_chat, submit_train_llm_chat,
                       test_log_probs, submit_log_probs,
                       test_softplus, submit_softplus,
                       test_dpo_loss, submit_dpo_loss,
                       test_train_dpo, submit_train_dpo,
                       test_eval_llm_chat, submit_eval_llm_chat,
                       test_eval_llm_dpo, submit_eval_llm_dpo)

os.environ["MUGRADE_HW"] = "Homework 6"
os.environ["MUGRADE_KEY"] = "kHhKRHek7TlCKeYQlpTo" ### Your key here

  Cloning https://github.com/locuslab/mugrade.git to /tmp/pip-req-build-jka_ytfm
  Running command git clone --filter=blob:none --quiet https://github.com/locuslab/mugrade.git /tmp/pip-req-build-jka_ytfm
  Resolved https://github.com/locuslab/mugrade.git to commit 717e300a5c2ddc0c729746946f8dc9f0d1c0ecea
  Preparing metadata (setup.py) ... done
  Created wheel for mugrade: filename=mugrade-1.3-py3-none-any.whl size=4113 sha256=25112bc304d042656450fa74223e1588bb1d9f8ecf804452c9823193bbd47b1d
  Stored in directory: /tmp/pip-ephem-wheel-cache-x5gclus8/wheels/34/d1/ce/8718af824354e197dbffca129b7b45c76dc4313b91b56b832c
Successfully built mugrade
--2026-04-25 11:52:35--  https://raw.githubusercontent.com/modernaicourse/hw6/refs/heads/main/hw6_tests.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, 

## Part 0 - Setting up the model

In this part, you'll want to copy the relevant model and training code from the previous lecture.  The following portions should all be _identical_ to the solutions in HW5, so we are including them with the same exact (local only) tests that were present in this previous homework, rather than new tests (and also leaving out the docstrings on the specific functions and classes ... the aim here is just to _exactly_ copy your implementation from the previous portion).

In [2]:
@mugrade.local_tests
class Linear(Module):
    """ Linear layer with no bias term.  The parameters of the layer are stored in a .weight Parameter"""
    def __init__(self, in_dim, out_dim):
        """
        Initialize a linear layer without a bias term.

        Inputs:
            in_dim : int - input feature dimension
            out_dim : int - output feature dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.weight = Parameter(torch.randn(out_dim, in_dim) * (2 / in_dim) ** 0.5)
        self.bias = Parameter(torch.zeros(out_dim))
        ### END YOUR CODE

    def forward(self, X):
        """
        Apply the linear layer to one or more input vectors.

        Input:
            X : torch.Tensor[float] (... x in_dim) - input tensor
        Output:
            torch.Tensor[float] (... x out_dim) - transformed tensor
        """
        ### BEGIN YOUR CODE
        return X @ self.weight.T + self.bias
        ### END YOUR CODE

@mugrade.local_tests
class Embedding(Module):
    def __init__(self, num_tokens, dim):
        """
        Initialize an embedding table over a fixed vocabulary.

        Inputs:
            num_tokens : int - vocabulary size
            dim : int - embedding dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.weight = Parameter(torch.randn(num_tokens ,dim))
        ### END YOUR CODE

    @property
    def shape(self):
        return self.weight.shape

    def zero_(self):
        self.weight.data.zero_()
        return self

    def __getitem__(self, idx):
      return self.weight[idx]

    def forward(self, Y):
        """
        Look up embeddings for an integer tensor of token ids.

        Input:
            Y : torch.Tensor[int] (...) - tensor of token indices in [0, num_tokens)
        Output:
            torch.Tensor[float] (... x dim) - embedding vectors for each token id
        """
        ### BEGIN YOUR CODE
        return self.weight[Y]
        ### END YOUR CODE

@mugrade.local_tests
def silu(x):
    ### BEGIN YOUR CODE
    return x * torch.sigmoid(x)
    ### END YOUR CODE

@mugrade.local_tests
def rms_norm(X, eps=1e-5):
    ### BEGIN YOUR CODE
    rms = torch.sqrt(torch.mean(X ** 2, dim=-1, keepdim=True) + eps)
    return X / rms
    ### END YOUR CODE

@mugrade.local_tests
def self_attention(Q,K,V, mask=None):
    ### BEGIN YOUR CODE
    d = Q.shape[-1]
    return torch.softmax(Q @ K.transpose(-2, -1) / d ** 0.5 + mask, dim=-1) @ V
    ### END YOUR CODE

@mugrade.local_tests
class MultiHeadAttentionKVCache(Module):
    def __init__(self, dim, n_heads, max_cache_size):
        """
        Initialize a multi-head self-attention layer with KV cache buffers.

        Inputs:
            dim : int - total embedding dimension
            n_heads : int - number of attention heads
            max_cache_size : int - maximum sequence length stored in the cache
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.wq = Linear(dim, dim)
        self.wk = Linear(dim, dim)
        self.wv = Linear(dim, dim)
        self.wp = Linear(dim, dim)

        self.head_dim = dim // n_heads
        self.n_heads = n_heads
        self.dim = dim

         # KV cache buffers (shape required by test)
        self.register_buffer(
            "k_cache",
            torch.zeros(1, max_cache_size, dim)
        )

        self.register_buffer(
            "v_cache",
            torch.zeros(1, max_cache_size, dim)
        )

        self.self_attention = self_attention
        ### END YOUR CODE

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply multi-head self-attention, optionally updating and using the KV cache.

        Inputs:
            X : torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the KV cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - attention output
        """
        ### BEGIN YOUR CODE
        batch, seq_len, dim = X.shape
        Q = self.wq(X)
        K = self.wk(X)
        V = self.wv(X)

        # ---- KV CACHE LOGIC ----
        if use_kv_cache:

            # store new keys/values
            self.k_cache[:, seq_pos:seq_pos+seq_len, :] = K
            self.v_cache[:, seq_pos:seq_pos+seq_len, :] = V

            # retrieve cached tokens
            K = self.k_cache[:, :seq_pos+seq_len, :]
            V = self.v_cache[:, :seq_pos+seq_len, :]

        total_len = K.shape[1]
        Q = Q.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1,2)
        K = K.view(batch, total_len, self.n_heads, self.head_dim).transpose(1,2)
        V = V.view(batch, total_len, self.n_heads, self.head_dim).transpose(1,2)

        heads = self.self_attention(Q, K, V, mask)
        heads = heads.transpose(1,2).contiguous().view(batch, seq_len, self.n_heads*self.head_dim)

        return self.wp(heads)
        ### END YOUR CODE


@mugrade.local_tests
class MLP(Module):
    def __init__(self, dim, ffn_dim):
        """
        Initialize the gated feed-forward network used in the transformer block.

        Inputs:
            dim : int - model dimension
            ffn_dim : int - hidden feed-forward dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.w1 = Linear(dim, ffn_dim)
        self.w2 = Linear(ffn_dim, dim)
        ### END YOUR CODE

    def forward(self, X):
        """
        Apply the gated feed-forward network to the input tensor.

        Input:
            X : torch.Tensor[float] (... x dim) - input tensor
        Output:
            torch.Tensor[float] (... x dim) - transformed tensor
        """
        ### BEGIN YOUR CODE
        return self.w2(silu(self.w1(X)))
        ### END YOUR CODE


@mugrade.local_tests
class TransformerBlock(Module):
    def __init__(self, dim, n_heads, ffn_dim, max_cache_size):
        """
        Initialize a transformer block with attention, normalization, and gated MLP layers.

        Inputs:
            dim : int - model dimension
            n_heads : int - number of attention heads
            ffn_dim : int - hidden feed-forward dimension
            max_cache_size : int - maximum sequence length stored in the attention cache
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.rms_norm = rms_norm
        self.mlp = MLP(dim, ffn_dim)
        self.attn = MultiHeadAttentionKVCache(dim, n_heads, max_cache_size)
        ### END YOUR CODE

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply one transformer block with residual connections.

        Inputs:
            X : torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the attention cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - transformed sequence embeddings
        """
        ### BEGIN YOUR CODE
        out_X = X + self.attn(self.rms_norm(X), mask=mask, seq_pos=seq_pos, use_kv_cache=use_kv_cache)
        return out_X + self.mlp(self.rms_norm(out_X))
        ### END YOUR CODE




@mugrade.local_tests
def cross_entropy_loss(logits, y):
    """
    Compute average cross entropy loss over a minibatch.

    Inputs:
        logits : ND torch.Tensor[float] (... x k) - predicted logits for each example
        y : (N-1)D torch.Tensor[int] (...) - desired class for each example
    Output:
        scalar torch.Tensor[float] - average cross entropy loss
    """
    ### BEGIN YOUR CODE
    num_classes = logits.shape[-1]
    logits = logits.reshape(-1, num_classes)
    y = y.reshape(-1)

    N = logits.shape[0]
    log_sum = torch.logsumexp(logits, dim=1)
    correct_labels = logits[torch.arange(N), y]
    return (log_sum - correct_labels).mean()
    ### END YOUR CODE


@mugrade.local_tests
class Adam:
    def __init__(self, params, lr=1e-3, betas = (0.9, 0.999), eps=1e-8):
        """
        Initialize Adam optimizer state for a set of parameters.

        Inputs:
            params : iterable[torch.nn.Parameter] - parameters to optimize
            lr : float - learning rate
            betas : tuple(float, float) - decay rates for first and second moments
            eps : float - numerical stability constant
        """
        ### BEGIN YOUR CODE
        self.params = list(params)
        self.lr = lr
        self.beta1 = betas[0]
        self.beta2 = betas[1]
        self.eps = eps

        # 🔹 Initialize moment estimates
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

        # 🔹 timestep
        self.t = 0
        ### END YOUR CODE

    def step(self):
        """
        Apply one Adam update to all stored parameters.
        """
        ### BEGIN YOUR CODE
        self.t += 1

        for i, p in enumerate(self.params):
            if p.grad is None:
                continue

            g = p.grad

            # 🔹 Update biased first moment
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g

            # 🔹 Update biased second moment
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (g * g)

            # 🔹 Bias correction
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            # 🔹 Parameter update
            p.data -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)
        ### END YOUR CODE

    def zero_grad(self):
        """
        Zero out gradients for all stored parameters when gradients exist.
        """
        ### BEGIN YOUR CODE
        for p in self.params:
          if p.grad is not None:
            p.grad.zero_()
        ### END YOUR CODE



Mugrade: Running local tests for function Linear():...
PASSED
Mugrade: Running local tests for function Embedding():...
PASSED
Mugrade: Running local tests for function silu():...
PASSED
Mugrade: Running local tests for function rms_norm():...
PASSED
Mugrade: Running local tests for function self_attention():...
PASSED
Mugrade: Running local tests for function MultiHeadAttentionKVCache():...
PASSED
Mugrade: Running local tests for function MLP():...
PASSED
Mugrade: Running local tests for function TransformerBlock():...
PASSED
Mugrade: Running local tests for function cross_entropy_loss():...
PASSED
Mugrade: Running local tests for function Adam():...
PASSED


In addition to the solutions above, there are some _very_ minor changes required to two functions or classes.  First, we're augmenting the LLM implementation from the previous homework to allow you to load and save models in a Llama-like format.  We're giving you the code for this, so you can just copy your solution to the respective functions, but you should not copy the _entire_ class from the previous assignments, but you should copy the `__init__()` and `forward()` functions and leave the other two functions as they are. The next two cells also have only local tests.

In [3]:
@mugrade.local_tests
class LLM(Module):
    def __init__(self, num_tokens, dim, n_heads, max_seq_len, ffn_dim, num_layers):
        """
        Initialize the simplified Llama 3 model used in this homework.

        Inputs:
            num_tokens : int - vocabulary size
            dim : int - model dimension
            n_heads : int - number of attention heads per layer
            max_seq_len : int - maximum supported sequence length
            ffn_dim : int - hidden feed-forward dimension in each block
            num_layers : int - number of transformer blocks
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.embedding = Embedding(num_tokens, dim)
        self.pos_embeddings = Embedding(max_seq_len, dim)

        self.dim = dim
        self.head_dim = dim // n_heads

        self.layers = ModuleList([
            TransformerBlock(dim, n_heads, ffn_dim, max_seq_len)
            for _ in range(num_layers)
        ])

        mask = torch.triu(
            torch.full((max_seq_len, max_seq_len), float("-inf")),
            diagonal=1
        )
        self.register_buffer("mask", mask)

        # self.linear1 = Linear(dim, ffn_dim)
        self.output = Linear(dim, num_tokens)
        self.norm = rms_norm
        ### END YOUR CODE

    def forward(self, tokens, seq_pos=0, use_kv_cache=False):
        """
        Apply the full language model to a batch of token sequences.

        Inputs:
            tokens : torch.Tensor[int] (batch_size x seq_len) - input token ids
            seq_pos : int - starting sequence position for positional embeddings and cached attention
            use_kv_cache : bool - whether to update and use cached keys and values
        Output:
            torch.Tensor[float] (batch_size x seq_len x num_tokens) - output logits
        """
        ### BEGIN YOUR CODE
        batch, seq_len = tokens.shape
        X_static = self.embedding(tokens)
        X_pos = [i for i in range(seq_pos, seq_pos+seq_len)]
        X_pos = self.pos_embeddings(X_pos)

        X = X_static + X_pos

        for block in self.layers:
          X = block(X, mask=self.mask[seq_pos:seq_pos+seq_len, :seq_pos+seq_len], seq_pos=seq_pos, use_kv_cache=use_kv_cache)

        return self.output(self.norm(X))
        ### END YOUR CODE

    def load_weights(self, filename):
        """ Load a model from a Llama-like checkpoint """
        checkpoint = torch.load(filename, map_location = self.embedding.weight.device)
        self.embedding.weight.data = checkpoint["tok_embeddings.weight"]
        self.pos_embeddings.data = checkpoint["pos_embeddings.weight"]
        self.output.weight.data = checkpoint["output.weight"]

        for i,layer in enumerate(self.layers):
            layer.attn.wq.weight.data = checkpoint[f"layers.{i}.attention.wq.weight"]
            layer.attn.wk.weight.data = checkpoint[f"layers.{i}.attention.wk.weight"]
            layer.attn.wv.weight.data = checkpoint[f"layers.{i}.attention.wv.weight"]
            layer.attn.wp.weight.data = checkpoint[f"layers.{i}.attention.wo.weight"]

            layer.mlp.w1.weight.data = checkpoint[f"layers.{i}.feed_forward.w1.weight"]
            layer.mlp.w2.weight.data = checkpoint[f"layers.{i}.feed_forward.w2.weight"]

    def save_weights(self, filename):
        """ Save a model to a Llama-like checkpoint."""
        checkpoint = {}
        checkpoint["tok_embeddings.weight"] = self.embedding.weight.detach().to(torch.bfloat16).cpu()
        checkpoint["pos_embeddings.weight"] = self.pos_embeddings.detach().to(torch.bfloat16).cpu()
        checkpoint["output.weight"] = self.output.weight.detach().to(torch.bfloat16).cpu()

        for i, layer in enumerate(self.layers):
            checkpoint[f"layers.{i}.attention.wq.weight"] = layer.attn.wq.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.attention.wk.weight"] = layer.attn.wk.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.attention.wv.weight"] = layer.attn.wv.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.attention.wo.weight"] = layer.attn.wp.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.feed_forward.w1.weight"] = layer.mlp.w1.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.feed_forward.w2.weight"] = layer.mlp.w2.weight.detach().to(torch.bfloat16).cpu()
        torch.save(checkpoint, filename)

Mugrade: Running local tests for function LLM():...
PASSED


You also will need to slightly revise the `generate()` function from the previous assignment to use an arbitrary token as the end token, since in chat contexts you will use the end assistant tag to stop generating further.

In [4]:
@mugrade.local_tests
def generate(model, prompt_tokens, tokenizer, eot_token=None, temp=0.7, max_tokens=500, verbose=True):
    """
    Autoregressively sample tokens from a language model using its KV cache.

    Inputs:
        model : Module - language model mapping token sequences to logits
        prompt_tokens : list[int] - initial prompt tokens
        tokenizer : object - tokenizer with decode()
        eot_token: int - index of token at which to top generation
        temp : float - sampling temperature
        max_tokens : int - maximum number of new tokens to generate
        verbose : bool - whether to print each generated token as it is sampled
    Output:
        list[int] - generated tokens, excluding the prompt tokens
    """
    ### BEGIN YOUR CODE
    tokens = torch.tensor([prompt_tokens], dtype=torch.long)

    seq_pos = 0
    generated = []

    for i in range(max_tokens):
        logits = model(tokens, seq_pos=seq_pos, use_kv_cache=True)

        next_logits = logits[0, -1] / temp
        probs = torch.softmax(next_logits, dim=-1)

        token = torch.multinomial(probs, 1)
        token_id = token.item()

        generated.append(token_id)

        if verbose:
            print(tokenizer.decode([token_id]), end="", flush=True)

        if token_id  == tokenizer.eot_token:
            break

        tokens = token.unsqueeze(0)
        if i == 0:
            seq_pos = len(prompt_tokens)
        else:
            seq_pos += 1
    return generated
    ### END YOUR CODE

Mugrade: Running local tests for function generate():...
PASSED


Finally, use the following code to download and load a pretrained model of this type.  This was trained with the same process as in the previous homework (in slightly more detail, it was trained using an 8-GPU parallel version of that homework's trainind code), but instead of the TinyStories dataset, it was trained on about 1.1 trillion tokens from the [FineWebEDU](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) dataset (this took about 3 days on 8 GPUs).  You won't need to do any of the re-training, but you should confirm that the model is able to generate text using the code below.

In [5]:
### Downlaod the models an datasets for the assignment
from huggingface_hub import hf_hub_download

repo = "zkolter/Chat-Tuning-Homework"
filenames = ["model_base.pth",
             "model_chat.pth",
             "params.json",
             "ultrachat_short.json",
             "ultrachat_dpo_neg.json",
             "ultrachat_dpo_pos.json"]

for filename in filenames:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="model", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_base.pth:   0%|          | 0.00/410M [00:00<?, ?B/s]

model_chat.pth:   0%|          | 0.00/410M [00:00<?, ?B/s]

params.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

ultrachat_short.json:   0%|          | 0.00/319M [00:00<?, ?B/s]

ultrachat_dpo_neg.json:   0%|          | 0.00/34.9M [00:00<?, ?B/s]

ultrachat_dpo_pos.json:   0%|          | 0.00/47.1M [00:00<?, ?B/s]

In [6]:
### load the model
with open("params.json", "rt") as f:
    params = json.load(f)
model = LLM(
    params["num_tokens"],
    params["dim"],
    params["n_heads"],
    params["max_seq_len"],
    params["ffn_dim"],
    params["n_layers"]
)
model.load_weights("model_base.pth")
model.float();

In [7]:
### generate some text
base_tokenizer = tiktoken.get_encoding("gpt2")
prompt = "In a surprising discovery,"
generate(model, base_tokenizer.encode(prompt,allowed_special="all"), base_tokenizer, temp=0.4, max_tokens=100);

ibli Tagsifiersosesriter LastTagsuzzifiersTags Lastacid Files Tags Tagsensus neighborribe763ilicifiersifiers Petersmeta763 commemor neighbor763 neighborribeifiers PlutoHM enrollmentissuesHM 1959ifiersifiersuniruckifiersalistifiersifieredarHMHMifiersedarHMordsHMifierslethal neighborHMifiers dioxide ::ifiersHMHM group Tags neighbor DocumentsHM commemorHMtomribeifier�� commercially Tags neighborHMifiersruction 625CopyrightHMHM lethalHMifiersifiersANI unambenaryigioys neighbor763ifier neighborHM763 neighbor

## Part I - Chat training via supervised finetuninng

Using the pretrained model above, in this part, you'll implement code to tune it for chat.  For this purpose, we're using a subject of the [Ultrachat 200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) dataset, a common dataset used for this kind of finetuning.  The subset of these chats included in the homework just filter the original data to only include those conversations that tokenize to 1024 tokens or fewer (the sequence length for our model.)


### Question 1 - Conversion to chat format

First, implement a funciton that converts a set of ultrachat-style messages to a text format with user and assistant tags.  Specifically, Ultrachat uses a message format like the following
```json
[
    {
        'role': 'user',
        'content': 'This is a user question.'
    },
    {
        'role': 'assistant',
        'content': 'This is an assistant response.'
    },
    {
        'role': 'user',
        'content': 'This is a followup user question.'
    },
    {
        'role': 'assistant',
        'content': 'This is a boring conversation.'
    }
]
```

You will want to convert this into a single text string that uses the tags `<USER>`, `</USER>`, `<ASSISTANT>`, and `</ASSISTANT>` to demarcate different portions of the conversation, with no additional newlines or whitespace.  For instance, the above content would become
```
<USER>This is a user question</USER><ASSISTANT>This is an assistant response.</ASSISTANT><USER>This is a followup user question.</USER><ASSISTANT>This is a boring conversation.</ASSISTANT>
```

In [8]:
@mugrade.submit_tests
def convert_to_chat_format(messages):
    """
    Convert a list of chat messages into a single tagged text string.

    Input:
        messages : list[dict[str, str]] - conversation with "role" and "content" keys
    Output:
        str - concatenated conversation using USER and ASSISTANT tags
    """
    ### BEGIN YOUR CODE
    result = ""
    for msg in messages:
      role, content = msg.items()
      result += f"<{msg['role'].upper()}>{msg['content']}</{msg['role'].upper()}>"
    return result
    ### END YOUR CODE

Mugrade: Submitting tests for function convert_to_chat_format():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


### Question 2 - Pretokenizing chat data

As in the previous homework, it's often convenient to create a pre-tokenized version of the dataset for more rapid iteration at training time.  This function should process data from the `ultrachat_short.json` file (which contains a list of conversations in the above format), and write another json file that contains a list of the same conversations, but where each conversation is represented itself as a list of integer tokens, as produced by applying the tokenizer to the chat-formatted version of the conversation.

As in the previous assignment, you'll want to tokenize the data using the function
```python
tokenizer.encode(text, allowed_special="all")
```

You should use the `json.load()` and `json.dump()` (with the optional argument `indent=4` if you want prettier outputs) to load and save the files.

In [9]:
import numpy as np

@mugrade.submit_tests
def pretokenize_chat(tokenizer, in_filename, out_filename):
    """
    Convert a chat dataset from json conversations to json token lists.

    Inputs:
        tokenizer : object - tokenizer with encode(text, allowed_special="all")
        in_filename : str - input json filename containing chat conversations
        out_filename : str - output json filename for tokenized conversations
    Output:
        None - writes tokenized conversations to out_filename
    """
    ### BEGIN YOUR CODE
    all_tokens = []
    with open(in_filename, "r", encoding="utf-8") as fread, open(out_filename, "w") as fwrite:
        super_messages = json.load(fread)
        for messages in super_messages:
          # print(messages)
          text = convert_to_chat_format(messages)
          # print(text)

          tokens = tokenizer.encode(text, allowed_special="all")
          # print(tokens)
          all_tokens.append(tokens)
        json.dump(all_tokens, fwrite)
    ### END YOUR CODE


Mugrade: Submitting tests for function pretokenize_chat():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


When you have implemented the function, you should be able to convert the full `ultrachat_short.json` file to this pretokenized format using the following code.  Note that this build the tokenizer to explicitly add our special tokens, so that these elements are always tokenized to their special ids.  This function takes some time to run, as it has to parse all ~90K of the included conversations.

In [10]:
base_tokenizer = tiktoken.get_encoding("gpt2")
tokenizer = tiktoken.Encoding(
    name="gpt2_chat",
    pat_str=base_tokenizer._pat_str,
    mergeable_ranks=base_tokenizer._mergeable_ranks,
    special_tokens={**base_tokenizer._special_tokens,
                    "<USER>": 50257,
                    "</USER>": 50258,
                    "<ASSISTANT>": 50259,
                    "</ASSISTANT>": 50300}
)

pretokenize_chat(tokenizer, "ultrachat_short.json", "ultrachat_tokenized.json")

### Question 3 - Chat masking

Write a function that will output a mask that characterizes which tokens should be trained on within a chat conversation.  For our purposes, we _only_ want to train on tokens that occur _after_ an `<ASSISTANT>` up through (and including) the `</ASSISTANT>` token.  So if the text was the following:
```
<USER>This is a user question</USER><ASSISTANT>This is an assistant response.</ASSISTANT><USER>This is a followup user question.</USER><ASSISTANT>This is a boring conversation.</ASSISTANT>
```
Then the mask should be `True` only for those tokens corresponding to the text `This is an assistant response.</ASSISTANT>` and `This is a boring conversation.</ASSISTANT>`


In [11]:
@mugrade.submit_tests
def get_loss_mask(tokens, tokenizer):
    """
    Build a boolean mask selecting assistant-response tokens for training.

    Inputs:
        tokens : list[int] - tokenized chat conversation
        tokenizer : object - tokenizer with special token ids for assistant tags
    Output:
        list[bool] - True for tokens after <ASSISTANT> through </ASSISTANT>
    """
    ### BEGIN YOUR CODE
    # print(tokens)
    result_bool = []
    start = False
    for id in tokens:
      if id == 93:
        start = True
        result_bool.append(False)

      elif id == 94:
        start = False
        result_bool.append(True)

      elif start:
        result_bool.append(True)
      else:
        result_bool.append(False)
    # print(result_bool)
    return result_bool
    ### END YOUR CODE

Mugrade: Submitting tests for function get_loss_mask():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


### Question 4 - A data loader for chat

Next, you'll build a data loader that loads (pretokenized) chat conversations and returns them as data suitable for training.  Unlike the data loader we built previously (which just operated on a binary storage file, so we could easily seek to different blocks of data), this one will just load the entire pretokenized json file and generate each batch on the fly.

As opposed to the data loader in the previous homework, which returned two tensors `x` and `y` where `y` was a shifted version of `x`, this loader will return _three_ element as follows
```python
for x,y,mask in loader:
    ...
```
where `x` and `y` are the same as before (`batch_size` by `seq_len` integer tensors), but where `mask` is a `batch_size` by `seq_len` _boolean_ tensor formed by the above `get_loss_mask()` function.

Although it isn't strictly necessary for supervised finetuning, in order for this same data loader to work for DPO, we're going to operate under the convention that each batch element consists of only a _single_ conversation, and you should zero-pad the end of the `x` and `y` outputs to ensure that they are all of length `seq_len`.  You will also zero-pad the end of `mask` with `False` entries, so that the zero-tokens at the end of the sequence are ignored by the loss function. You should also drop incomplete batches, i.e., batches which have size < `batch_size`.

In [12]:
@mugrade.submit_tests
class DataLoaderChat:
    def __init__(self, filename, seq_len, batch_size, tokenizer, device="cpu"):
        """
        Initialize a chat data loader backed by a tokenized json file.

        Inputs:
            filename : str - json filename containing tokenized conversations
            seq_len : int - number of tokens per input sequence
            batch_size : int - number of conversations per minibatch
            tokenizer : object - tokenizer used to build assistant-response masks
            device : str - device on which to place each minibatch tensor
        """
        ### BEGIN YOUR CODE
        self.state = 0
        self.content = json.load(open(filename, "r", encoding="utf-8"))
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.tokenizer = tokenizer
        self.device = device
        ### END YOUR CODE

    def __iter__(self):
        """
        Reset iteration state and return the iterator object.

        Output:
            DataLoaderChat - iterator over chat minibatches
        """
        ### BEGIN YOUR CODE
        self.state = 0
        return self
        ### END YOUR CODE

    def __next__(self):
        """
        Return the next chat minibatch with masked targets.

        Output:
            tuple(torch.Tensor, torch.Tensor, torch.Tensor) - input tokens, next-token targets, and boolean loss mask
        """
        ### BEGIN YOUR CODE
        if self.state >= len(self.content):
          raise StopIteration


        X = []
        Y = []
        M = []

        batch_data = self.content[self.state:self.state+self.batch_size]
        self.state += self.batch_size
        for data in batch_data:
          tokens = data[:self.seq_len+1]
          if len(tokens) < self.seq_len + 1 :
            tokens = tokens + [0]*(self.seq_len + 1 - len(tokens))
          x = tokens[:-1]
          # print(x)
          y = tokens[1:]
          # print(y)
          mask = get_loss_mask(y, self.tokenizer)
          X.append(x)
          Y.append(y)
          M.append(mask)

        input_batch = torch.tensor(X, dtype=torch.long).to(self.device)
        target_batch = torch.tensor(Y, dtype=torch.long).to(self.device)
        mask_batch = torch.tensor(M, dtype=torch.bool).to(self.device)
        return  (input_batch, target_batch, mask_batch)
        ### END YOUR CODE

Mugrade: Submitting tests for function DataLoaderChat():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Grader test 6 passed
Grader test 7 passed
Grader test 8 passed
Grader test 9 passed


### Question 5 - Chat training loop

Write a function to finetune a model based upon chat training.  This should use essentially the same logic as your LLM training code in homework 5, with the only difference being the application of the mask to the relevant data.  Note that it is specifically helpful to use PyTorch's boolean indexing for this purpose.  That is, if e.g., `y` is a tensor of some size and `mask` is a boolean tensor of the same size then
```python
y[mask]
```
will return a 1D tensor _only_ of the entires in `y` corresponding to `True` entries in `mask`.  The same can be used to index based upon only the leading dimensions of a higher-order tensor.  You can see this e.g., in the following code:
```python
A = torch.randn(3,4,5)
mask = torch.randn(3,4) > 0
print(A[mask])
```
which will be a `n` by 5 dimensionsal tensor, where `n` is the total number of `True` entires in mask.

We also pass an additional `max_iter` parameter to keep things simple, where the training loop should return after `max_iter` iterations.

In [13]:
@mugrade.submit_tests
def train_llm_chat(model, chat_loader, opt, max_iter = None):
    """
    Run one pass of supervised chat finetuning with a masked next-token loss.

    Inputs:
        model : Module - language model mapping token sequences to logits
        chat_loader : iterable - yields minibatches of chat inputs, targets, and masks
        opt : optimizer - object with zero_grad() and step() methods
        max_iter : int or None - maximum number of minibatches to process
    Output:
        None - updates model parameters in place
    """
    ### BEGIN YOUR CODE
    model.train()
    counter = 0
    for x, y, m in chat_loader:
      opt.zero_grad()
      counter += 1
      if max_iter and max_iter < counter:
        break
      y_pred = model(x)
      loss = cross_entropy_loss(y_pred[m], y[m])
      loss.backward()
      opt.step()
    ### END YOUR CODE

Mugrade: Submitting tests for function train_llm_chat():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


If you implemented the function above correctly, then the following code should perform a few iterations of chat tuning on the pretrained model.  Note that we recommend you finetune the model with 32 bit floating point precision (instead of the 16 bit precision you used for pretraining in Homework 5).

In [14]:
# loader = DataLoaderChat("ultrachat_tokenized.json", 1024, 2, tokenizer, device="cpu")
# model.load_weights("model_base.pth")
# model.float().cpu();

device = "cuda"  # or torch.device("cuda")

loader = DataLoaderChat(
    "ultrachat_tokenized.json",
    1024,
    2,
    tokenizer,
    device=device
)

model.load_weights("model_base.pth")

model = model.float().to(device)

In [ ]:
opt = Adam(model.parameters(), lr=2e-5, betas=(0.9, 0.95))
train_llm_chat(model, loader, opt, max_iter=10)

Note that we're training a model that is already fine-tuned, so you won't see a huge loss decrease, but the following code will test you system (the test cases mainly look to see if the chat tokens, especially the end assistant tag have increased in probability).

In [15]:
@mugrade.submit_tests
def eval_llm_chat():
    return model

Mugrade: Submitting tests for function eval_llm_chat():


AssertionError: 

You can train a larger chat model (again using a GPU, which would be necessary for this), using the following code.

In [ ]:
loader = DataLoaderChat("ultrachat_tokenized.json", 1024, 32, tokenizer, device="cuda")
model.load_weights("model_base.pth")
model.float().cuda(); # keep in float32 format for finetuning

In [ ]:
opt = Adam(model.parameters(), lr=2e-5, betas=(0.9, 0.95))
train_llm_chat(model, loader, opt)

You can try generating text from the model using the following code.

In [ ]:
prompt = "<USER>What is the capital of the United States?</USER><ASSISTANT>"
generate(model, tokenizer.encode(prompt,allowed_special="all"), tokenizer, eot_token=50300, temp=0.4);

 Al I Alrantigi Files latest aff Last Files inertigi Last Last Last Last eHM sensitive;n only to is Last no Group isn no is Last how ton no no is is is is is those is does is that perform Under is is is is is is is is is is is is Group is is is is is is is is is is is is is is is is is is is is is is is is is is is is because is is is is is is is is is is is is is is is is is is is is great is Group is is is is. is is is is is is is Last! is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is fact Group is is is is re is is is is is is is is is is is is use is is is is is is is is is is is is is is is is is is is fact is is remember is fact is. fact is is is fact sign is. fact sign what is is is is is is fact Last! to is remember is. is is is. is is is! is is.! is! is is is is is is. sign is fact is is is remember sign is make remember is fact the is is is is is is not N sign is is is is. sign is is sign is is is is is i is is is is sign is is is is i

## Part II - Direct preference optimization

In this section of the homework you'll implement the direct preference optimization (DPO) algorithm to further finetune the chat model with human preference data.  If you were able to finetune the complete chat model on the entire dataset, you can use that model as a starting point, but we also include a `model_chat.json` (which we already trained on the full dataset) that you can use as the starting poing.

### Question 6 - Log probability calculation

For DPO, we will need the ability to compute the sum of log probabilities (i.e., negative cross entropy losses), not just averaged over the entire batch, but for each element in the batch separately.  Notationally, we will use the convention of computing the log proability instead of negating the cross-entropy loss, because this is more common in presentation of DPO, and it's helpful in better understanding the connection between the two perspectives.

Implement the following function, which takes similar inputs to the cross entropy loss function, but which instead computes the _sum_ of the (masked) log probabilities (negative cross entropy loss, computed in the same manner), over each element of the batch separately.

In [21]:
@mugrade.submit_tests
def log_probs(logits, y, mask):
    """
    Compute masked sequence log probabilities for each batch element.

    Inputs:
        logits : torch.Tensor[float] (batch_size x seq_len x num_tokens) - predicted logits
        y : torch.Tensor[int] (batch_size x seq_len) - desired next-token ids
        mask : torch.Tensor[bool] (batch_size x seq_len) - mask selecting tokens to include
    Output:
        torch.Tensor[float] (batch_size,) - summed masked log probabilities per example
    """
    ### BEGIN YOUR CODE
    import torch
    import torch.nn.functional as F

    # convert logits → log-probabilities
    logp = F.log_softmax(logits, dim=-1)   # [B, T, V]

    # pick log-prob of correct tokens
    # gather along vocab dimension
    token_logp = logp.gather(dim=2, index=y.unsqueeze(-1)).squeeze(-1)  # [B, T]

    # apply mask (ignore False positions)
    token_logp = token_logp * mask   # [B, T]

    # sum over sequence → per batch element
    seq_logp = token_logp.sum(dim=1)  # [B]

    return seq_logp
    ### END YOUR CODE

Mugrade: Submitting tests for function log_probs():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


### Question 7 - DPO Loss

Next, implement a function that computes the DPO loss, which is defined as the following.  Given a triple of examples $x,y^+, y^-$ (prompt, positive completion, and negative completion), the loss is given by
$$
L_{\text{DPO}} = \text{softplus}\left (-\log\frac{p(y^+ | x)}{p_{\text{ref}}(y^+|x)} + \log\frac{p(y^- | x)}{p_{\text{ref}}(y^-|x)}, \beta \right )
$$
where $p(\cdot)$ denotes probability under the model being trained, and $p_\mathrm{ref}$ is a reference model that remains fixed (in this case, we'll use the initial model before DPO as a fixed reference), and where the softplus function is defined as
$$
\text{softplus}(x,\beta) = \log\left (1+e^{\beta x} \right ).
$$
[The softplus function acts as a softer version of a ReLU-like function, and capture the intuition that we want to minimize the term inside the DPO loss only when the inner term is positive, i.e., the probability of the negative completion is larger than the probability of the positive one).

You shouldn't implement the softplus function using `log` or `exp` functions directly, but instead use the `torch.logaddexp()` function to compute it in a more numerically stable fashion.

For efficiency and to ensure that you aren't actually modifying the reference model, you should ensure that all use of the reference model is done within a `torch.no_grad()` both, similar to how you implemented this in the optimizer.

In [23]:
@mugrade.submit_tests
def softplus(x, beta):
    """
    Compute the beta-scaled softplus function elementwise.

    Inputs:
        x : torch.Tensor[float] (...) - input tensor
        beta : float - scaling factor inside the softplus
    Output:
        torch.Tensor[float] (...) - softplus(beta * x)
    """
    ### BEGIN YOUR CODE
    return torch.log(1+torch.exp(beta*x))
    ### END YOUR CODE

Mugrade: Submitting tests for function softplus():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


In [28]:
@mugrade.submit_tests
def dpo_loss(model, model_ref, xp,yp,maskp, xn, yn, maskn, beta):
    """
    Compute the DPO loss for paired preferred and dispreferred completions.

    Inputs:
        model : Module - model being optimized
        model_ref : Module - fixed reference model
        xp : torch.Tensor[int] (batch_size x seq_len) - preferred input tokens
        yp : torch.Tensor[int] (batch_size x seq_len) - preferred next-token targets
        maskp : torch.Tensor[bool] (batch_size x seq_len) - preferred loss mask
        xn : torch.Tensor[int] (batch_size x seq_len) - dispreferred input tokens
        yn : torch.Tensor[int] (batch_size x seq_len) - dispreferred next-token targets
        maskn : torch.Tensor[bool] (batch_size x seq_len) - dispreferred loss mask
        beta : float - inverse temperature for the DPO objective
    Output:
        torch.Tensor[float] (batch_size,) - DPO loss for each preference pair
    """
    ### BEGIN YOUR CODE
    import torch
    import torch.nn.functional as F

    # --- current model ---
    logp_p = log_probs(model(xp), yp, maskp)   # [B]
    logp_n = log_probs(model(xn), yn, maskn)

    # --- reference model ---
    with torch.no_grad():
        logp_p_ref = log_probs(model_ref(xp), yp, maskp)
        logp_n_ref = log_probs(model_ref(xn), yn, maskn)

    # differences
    model_diff = logp_p - logp_n
    ref_diff   = logp_p_ref - logp_n_ref

    # DPO loss using softplus
    loss = softplus(-(model_diff - ref_diff), beta)  # [B]

    return loss
    ### END YOUR CODE

Mugrade: Submitting tests for function dpo_loss():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


### Question 8 - DPO training loop

Finally, we'll implement the training function for DPO.  To start, we'll note that this training loop will use _two_ data loaders, capturing the positive and negative examples (in the exact same order, so that you can iterate over them simultaneously).  The positive examples here were included in Ultrachat 200k, while the negative examples came using the same prompts to generate responses from our original chat-tuned model.

In [29]:
### pre-tokenize the two
pretokenize_chat(tokenizer, "ultrachat_dpo_neg.json", "ultrachat_neg_tokenized.json")
pretokenize_chat(tokenizer, "ultrachat_dpo_pos.json", "ultrachat_pos_tokenized.json")

Now, implement the following `train_dpo` function.  This should follow the same rough approach as the other training loops, but use the DPO loss instead of a cross entropy loss.  

In [51]:
@mugrade.submit_tests
def train_dpo(model, model_ref, loader_pos, loader_neg, opt, beta=0.1, max_iter = None):
    """
    Run one pass of DPO finetuning over paired positive and negative minibatches.

    Inputs:
        model : Module - model being optimized
        model_ref : Module - fixed reference model
        loader_pos : iterable - yields preferred chat minibatches
        loader_neg : iterable - yields dispreferred chat minibatches
        opt : optimizer - object with zero_grad() and step() methods
        beta : float - inverse temperature for the DPO objective
        max_iter : int or None - maximum number of minibatches to process
    Output:
        None - updates model parameters in place
    """
    ### BEGIN YOUR CODE
    counter = 0
    model.train()
    for (xp, yp, maskp), (xn, yn, maskn) in zip(loader_pos, loader_neg):
      if max_iter is not None and max_iter <= counter:
        break

      counter += 1
      opt.zero_grad()
      loss = dpo_loss(model, model_ref, xp, yp, maskp, xn, yn, maskn, beta)
      loss = loss.mean()
      loss.backward()
      opt.step()
    ### END YOUR CODE

Mugrade: Submitting tests for function train_dpo():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


### Question 9 - Training a DPO model

Finally, we can put all these functions together to train a DPO model.  We will start by initializing the data loaders and model.  You can dirctly use the model you trained above as a starting point, but for ease of use, we recommend that you use a version of the model that we fine-tuned using the entire data set above, stored in `model_chat.json`.  To start, you can train a small version on CPU.

In [52]:
loader_neg = DataLoaderChat("ultrachat_neg_tokenized.json", 1024, 2, tokenizer, device="cpu")
loader_pos = DataLoaderChat("ultrachat_pos_tokenized.json", 1024, 2, tokenizer, device="cpu")
model.load_weights("model_chat.pth") # comment out if you want to use your own model
model.float().cpu();
# shallow copy shares structure
model_ref = copy.copy(model)
model_ref.load_state_dict(copy.deepcopy(model.state_dict()))
model_ref = model_ref.cpu().float();

In [ ]:
opt = Adam(model.parameters(), lr=1e-6, betas=(0.9, 0.95))
train_dpo(model, model_ref, loader_pos, loader_neg, opt, max_iter=10)

You can test the trained model using the following code.

In [ ]:
@mugrade.local_tests
def eval_llm_dpo():
    return model

And finally, if you want you can train a GPU version with the entire dataset.

In [ ]:
### Set up data loaders for large DPO run
loader_neg = DataLoaderChat("ultrachat_neg_tokenized.json", 1024, 16, tokenizer, device="cuda")
loader_pos = DataLoaderChat("ultrachat_pos_tokenized.json", 1024, 16, tokenizer, device="cuda")
model.load_weights("model_chat.pth") # comment out if you want to use your own model
model.float().cuda();
model_ref = copy.deepcopy(model);

In [ ]:
### Train DPO
opt = Adam(model.parameters(), lr=1e-6, betas=(0.9, 0.95))
train_dpo(model, model_ref, loader_pos, loader_neg, opt)

In [ ]:
### Test some prompt generation
prompt = "<USER>Write a poem about flowers.</USER><ASSISTANT>"
generate(model, tokenizer.encode(prompt,allowed_special="all"), tokenizer, eot_token=50300, temp=0.4);